In [0]:
%sql
create or replace temp view vw_dim_client as(
  with dedup as (
    select
       client_id
      ,client_name
      ,row_number() over (
        partition by 
           client_id 
        order by 
           sale_date desc
          ,ingestion_timestamp desc
      )                                                           as order_by_date
    from workspace.pj_sales.tb_sales_silver
  )
  select
     client_id
    ,client_name
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as valid_from
    ,null                                                           as valid_to
  from dedup
  where order_by_date = 1
)

In [0]:
%sql
create or replace temp view vw_dim_client_merge as
select
  vc.*,
  'u' as action
from vw_dim_client vc -- Aqui o registro serve para ser atualizado
union all
select
  vc.*,
  'i' as action
from vw_dim_client vc -- Aqui o registro serve para ser inserido

In [0]:
%sql
merge into workspace.pj_sales.tb_dim_client_gold tc
using vw_dim_client_merge vcm
on  tc.client_id = vcm.client_id
and tc.valid_to is null
and vcm.action = 'u'
when matched and vcm.action = 'u' then
  update set
    tc.valid_to = vcm.valid_from
when not matched and vcm.action = 'i' then
  insert (
     client_id
    ,client_name
    ,valid_from
    ,valid_to
  )
  values (
     vcm.client_id
    ,vcm.client_name
    ,vcm.valid_from
    ,vcm.valid_to
  )
